# Module 02: Running for Free -- LLM Provider Strategy

**Estimated time: 45 minutes**

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Switch between LLM providers using only environment variables
- Explain the trade-offs between mock, Ollama, Groq, and Bedrock
- Understand the factory pattern for abstracting infrastructure
- Run the full pipeline against a real LLM for $0

## The Invisible Barrier

Most AI tutorials begin with: **'First, get an API key.'**

This creates an invisible barrier:
- Students without credit cards can't follow along
- Professionals on restricted work accounts can't experiment
- Nobody wants to pay for experiments that might not work

This project is designed around a different principle:

> **You should be able to learn the entire AI engineering stack for $0.**
> Pay only when you're in production -- or for that one demo that matters.

The `LLM_PROVIDER` environment variable lets you progress through a cost ladder:

| Stage | Provider | Cost | When |
|---|---|---|---|
| Learning | Mock | $0 | Building logic, CI/CD |
| Development | Ollama | $0 | Testing with real LLMs locally |
| Sharing | Groq | $0 (rate limited) | Demos, sharing with others |
| Production | Bedrock | ~$0.03-0.10/run | Interviews, live systems |

In [ ]:
import sys
import os
import inspect

sys.path.insert(0, os.path.abspath('..'))

# Read the factory source
from src.llm import factory
print('=== LLM Factory Source ===')
print(inspect.getsource(factory))

## The Factory Pattern

```python
def get_llm(provider=None):
    provider = provider or os.getenv('LLM_PROVIDER', 'ollama')
    if provider == 'ollama':    return ChatOllama(...)
    elif provider == 'groq':    return ChatGroq(...)
    elif provider == 'bedrock': return ChatBedrock(...)
```

Every agent in this system calls `get_llm()` -- never `ChatOllama()` or `ChatBedrock()` directly.

This means:
- The agent code **never changes** when you switch providers
- You switch providers by changing **one environment variable**
- Adding a new provider (OpenAI, Azure, Anthropic direct) is adding **one elif**

This is production AI architecture. Abstracting the LLM behind an interface
keeps business logic portable across provider changes.

In [ ]:
# Mock mode -- zero API calls, instant, always works
# Perfect for: building logic, CI/CD, initial learning
from src.pipeline.mock import MOCK_DECISIONS

alert_id = 'ALERT-2024-001'
decision = MOCK_DECISIONS[alert_id]

print('Mock decision for ' + alert_id + ':')
print('  Severity:   ' + decision['severity'])
print('  Confidence: ' + str(int(decision['confidence']*100)) + '%')
print('  Techniques: ' + str(decision['mitre_techniques']))
print()
print('No API call made. No cost. Instant.')

In [ ]:
# How to switch providers -- set an env var, that's it
current = os.getenv('LLM_PROVIDER', 'ollama (default)')
print('Current provider: ' + current)
print()
print('To use Ollama (free, local):')
print('  1. brew install ollama   (Mac) or see ollama.ai')
print('  2. ollama pull llama3.1')
print('  3. Add LLM_PROVIDER=ollama to your .env')
print('  4. python demo.py --live')
print()
print('To use Groq (free cloud):')
print('  1. Get free key at console.groq.com  (no credit card)')
print('  2. Add GROQ_API_KEY=your_key to your .env')
print('  3. Add LLM_PROVIDER=groq to your .env')
print('  4. python demo.py --live')

## Cost Reality Check: Bedrock

When you're ready for Bedrock (production-grade), here's the math:

**Claude 3 Haiku on Bedrock:**
- Input:  $0.00025 per 1,000 tokens
- Output: $0.00125 per 1,000 tokens

**Per demo run (5 alerts, ~2,000 tokens each):**
- Input: ~10,000 tokens = $0.0025
- Output: ~3,000 tokens = $0.00375
- **Total: ~$0.006 per run** (less than one cent)

For 100 development runs: **$0.60 total.**

The 'AI is expensive' perception comes from production scale, not development.
For learning and demos, cloud LLMs are effectively free.

**The right progression:**
Mock -> Ollama -> Groq -> Bedrock (only when you need production quality)

## Exercises

### Beginner
Run `python demo.py` (mock mode). Then set up Groq:
1. Get a free key at [console.groq.com](https://console.groq.com)
2. `cp .env.example .env` and fill in your key
3. Run `python demo.py --live`
4. Compare the Groq reasoning to mock reasoning for the same alert. What's different?

### Intermediate
Add OpenAI as a provider to `src/llm/factory.py`:
- `elif provider == 'openai': return ChatOpenAI(...)` from `langchain_openai`
- You don't need an actual key to write and test this code
- Run `pytest tests/` to verify nothing broke

### Advanced
Build a cost tracking wrapper:
- Wrap the LLM calls to count tokens per provider using LangChain callbacks
- Calculate estimated cost based on provider pricing
- Print a report after each demo: 'Total tokens: X | Estimated cost: $Y'

---

## Module 02 Complete

You've:
- Understood the provider ladder (mock -> Ollama -> Groq -> Bedrock)
- Read the factory pattern implementation
- Seen why provider abstraction matters for long-term maintainability

**Next: [Module 03 -- Evaluation](03_evaluation.ipynb)**
Teach AI to grade its own work.